In [41]:
import tensorflow as tf
import os, datetime
import tensorflow_recommenders as tfrs
from google.cloud import bigquery
import numpy as np
import pandas as pd
import pprint
from typing import Dict, Text
import logging
import faiss
from keras_tuner import HyperModel, HyperParameters
from keras_tuner.tuners import RandomSearch
from keras_tuner import Objective
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [2]:
# Check for GPU availability and set memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logger.info(f"GPUs available: {len(gpus)}")
    except RuntimeError as e:
        logger.warning(e)
else:
    logger.info("No GPUs available.")

2025-01-20 13:36:49,452 - INFO - No GPUs available.


In [3]:
# Define the BigQuery table and project details
PROJECT_ID = 'oolola'
DATASET_ID = 'movie_data'
TABLE_ID   = 'preprocessed_data'
timestamp  = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
output_dir = 'gs://movie-data-1/trained-model'
# Function to load movies from BigQuery
def load_movies_bq():
    try:
        client = bigquery.Client(project=PROJECT_ID)
        query = f"""
        SELECT title, genres
        FROM `{PROJECT_ID}.{DATASET_ID}.preprocessed_movies`
        """
        query_job = client.query(query)
        return query_job.to_dataframe()
    except Exception as e:
        logger.error(f"Error loading movies from BigQuery: {e}")
        raise
# Function to load ratings from BigQuery
def load_ratings_bq():
    try:
        client = bigquery.Client(project=PROJECT_ID)
        query = f"""
        SELECT user_id, title, rating
        FROM `{PROJECT_ID}.{DATASET_ID}.ratings_with_titles`
        """
        query_job = client.query(query)
        return query_job.to_dataframe()
    except Exception as e:
        logger.error(f"Error loading ratings from BigQuery: {e}")
        raise

# Function to get movie titles based on movie IDs
def get_titles(movie_ids: list):
    try:
        client = bigquery.Client(project=PROJECT_ID)
        # Convert the list of movie IDs to a comma-separated string
        movie_ids_str = ', '.join(map(str, movie_ids))
        query = f"""
        SELECT title
        FROM `{PROJECT_ID}.{DATASET_ID}.genres`
        WHERE movie_id IN ({movie_ids_str})
        """
        query_job = client.query(query)
        return query_job.to_dataframe()['title'].tolist()
    except Exception as e:
        logger.error(f"Error getting titles from BigQuery: {e}")
        raise

In [4]:
movies_bq = load_movies_bq()
movies_dict = {key: list(value) for key, value in movies_bq[['title', 'genres']].to_dict(orient='list').items()}

In [5]:
ratings_bq = load_ratings_bq()
ratings_dict = {key: list(value) for key, value in ratings_bq[['title', 'user_id', 'rating']].to_dict(orient='list').items()}

In [6]:
ratings = tf.data.Dataset.from_tensor_slices(ratings_dict)

2025-01-20 13:37:02.729089: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [7]:
for x in ratings.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'rating': 2.0, 'title': b'Initiation', 'user_id': 56703}


In [8]:
movies = tf.data.Dataset.from_tensor_slices(movies_dict)

In [9]:
for x in movies.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'genres': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]),
 'title': b'Siberian Sniper'}


In [10]:
ratings = ratings.map(lambda x: {
            "title": x["title"],
            "user_id": x["user_id"],
            "rating": x["rating"]
        })
movies = movies.map(lambda x: {
    "title": x["title"],
    "genres": x["genres"]
})

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


2025-01-20 13:37:03,839 - WARNING - From /Users/dmmckinn/repos/movie_recommender/.venv/lib/python3.10/site-packages/tensorflow/python/autograph/pyct/static_analysis/liveness.py:83: Analyzer.lamba_check (from tensorflow.python.autograph.pyct.static_analysis.liveness) is deprecated and will be removed after 2023-09-23.
Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


In [11]:
# Create a dictionary from the movies dataset
movies_dict = {movie["title"].numpy().decode('utf-8'): movie["genres"].numpy() for movie in movies}

def combine_datasets(rating):
    def lookup_genres(title):
        title_str = title.numpy().decode('utf-8')  # Convert to numpy and decode the title from bytes to string
        return movies_dict.get(title_str, [0] * 19)

    genres = tf.py_function(
        func=lookup_genres,
        inp=[rating["title"]],
        Tout=tf.int32
    )
    genres.set_shape([19])
    rating["genres"] = genres
    return rating

combined_dataset = ratings.map(combine_datasets)

In [12]:
# print one example of combined dataset
for x in combined_dataset.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'genres': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0],
      dtype=int32),
 'rating': 2.0,
 'title': b'Initiation',
 'user_id': 56703}


In [13]:
combined_dataset = combined_dataset.map(lambda x: {
    "title": x["title"],
    "user_id": x["user_id"],
    "genres": x["genres"],
    "rating": x["rating"]
}, num_parallel_calls=tf.data.AUTOTUNE)

In [45]:
# Set the seed for reproducibility
tf.random.set_seed(42)

# Shuffle the dataset with a large buffer size
# Ensure the buffer size is large enough to cover randomness but not too large to exhaust memory
shuffle_buffer_size = 200000  # Smaller buffer size for faster shuffling
shuffled = combined_dataset.shuffle(buffer_size=shuffle_buffer_size, seed=42, reshuffle_each_iteration=True)

# Calculate relative proportions for splitting
train_ratio = 0.8

ds_length = int(tf.data.experimental.cardinality(shuffled).numpy())
print(f"Length of the dataset: {ds_length}")

# Define the split function for large datasets
def split_dataset(dataset, train_ratio):
    # Determine split points
    trainds = dataset.take(int(train_ratio * ds_length))
    testds = dataset.skip(int(train_ratio * ds_length))

    return trainds, testds

# Perform the split
trainds, testds = split_dataset(shuffled, train_ratio)

# Optimize performance with prefetching
train_batch_size =  128
eval_test_batch_size = 64

# Optimize datasets with batching, caching, and prefetching
trainds = trainds.batch(train_batch_size).cache().prefetch(tf.data.AUTOTUNE)
testds = testds.batch(eval_test_batch_size).cache().prefetch(tf.data.AUTOTUNE)

Length of the dataset: 174490


In [15]:
titles = movies.batch(100000).map(lambda x: x["title"])
userIds = ratings.batch(1000000).map(lambda x: x["user_id"])
genres = movies.batch(100000).map(lambda x: {
        "title": x["title"],
        "genres": x["genres"]
    })

In [16]:
unique_titles = np.unique(np.concatenate(list(titles)))
unique_userIds = np.unique(np.concatenate(list(userIds)))
unique_genres = [
            'Action',
            'Adventure',
            'Animation',
            'Children',
            'Comedy',
            'Crime',
            'Drama',
            'Documentary',
            'Fantasy',
            'Film-Noir',
            'Horror',
            'IMAX',
            'Musical',
            'Mystery',
            'Romance',
            'Sci-Fi',
            'Thriller',
            'War',
            'Western'
        ]

In [17]:
class RecommendationModel(tfrs.Model):
    def __init__(self, user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, rating_weight=1.0, retrieval_weight=1.0):
        super().__init__()
        self.user_model = user_model
        self.movie_model = movie_model
        self.genre_model = genre_model
        self.rating_model = rating_model
        self.rating_task = rating_task
        self.retrieval_task = retrieval_task
        self.rating_weight = rating_weight
        self.retrieval_weight = retrieval_weight

    def call(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        user_embeddings = self.user_model(features["user_id"])
        movie_embeddings = self.movie_model(features["title"])
        genre_embeddings = self.genre_model(features["genres"])
        rating_predictions = self.rating_model([features["user_id"], features["title"], features["genres"]])
        return user_embeddings, movie_embeddings, genre_embeddings, rating_predictions

    def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
        ratings = features.pop("rating")
        user_embeddings, movie_embeddings, genre_embeddings, rating_predictions = self(features)
        rating_loss = self.rating_task(labels=ratings, predictions=rating_predictions)
        retrieval_loss = self.retrieval_task(user_embeddings, movie_embeddings)
        return (self.rating_weight * rating_loss
                + self.retrieval_weight * retrieval_loss)

In [26]:
# Function to create the model
def create_model(unique_user_ids, unique_movie_ids, num_genres, rating_weight=1.0, retrieval_weight=1.0):
    embedding_dimension = 128
    user_input = tf.keras.layers.Input(shape=(), dtype=tf.int32, name='user_id')
    movie_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='title')
    genre_input = tf.keras.layers.Input(shape=(num_genres,), dtype=tf.float32, name='genres')

    user_lookup = tf.keras.layers.IntegerLookup(vocabulary=unique_user_ids, mask_token=None)
    movie_lookup = tf.keras.layers.StringLookup(vocabulary=unique_titles, mask_token=None)

    user_embedding = tf.keras.layers.Embedding(len(unique_user_ids) + 1, embedding_dimension)(user_lookup(user_input))
    movie_embedding = tf.keras.layers.Embedding(len(unique_movie_ids) + 1, embedding_dimension)(movie_lookup(movie_input))
    genre_embedding = tf.keras.layers.Dense(embedding_dimension)(genre_input)

    concatenated_embeddings = tf.concat([user_embedding, movie_embedding, genre_embedding], axis=1)

    dense_1 = tf.keras.layers.Dense(256, activation="relu")(concatenated_embeddings)
    dropout_1 = tf.keras.layers.Dropout(0.5)(dense_1)
    dense_2 = tf.keras.layers.Dense(128, activation="relu")(dropout_1)
    dropout_2 = tf.keras.layers.Dropout(0.5)(dense_2)
    rating_output = tf.keras.layers.Dense(1)(dropout_2)


    user_model = tf.keras.Model(inputs=user_input, outputs=user_embedding)
    movie_model = tf.keras.Model(inputs=movie_input, outputs=movie_embedding)
    genre_model = tf.keras.Model(inputs=genre_input, outputs=genre_embedding)
    rating_model = tf.keras.Model(inputs=[user_input, movie_input, genre_input], outputs=rating_output)

    metrics = tfrs.metrics.FactorizedTopK(
        candidates=tf.data.Dataset.from_tensor_slices(unique_movie_ids).batch(128).map(movie_model)
    )
    rating_task = tfrs.tasks.Ranking(
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=[tf.keras.metrics.RootMeanSquaredError()],
    )
    retrieval_task = tfrs.tasks.Retrieval(
        metrics=metrics
    )

    model = RecommendationModel(user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, rating_weight, retrieval_weight)
    lr_schedule_ed = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=0.1,
        decay_steps=545,
        decay_rate=0.9,
        staircase=True
    )

    # lr_schedule_pcd = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
    #     boundaries=[2200, 3000, 3500],
    #     values=[0.1, 0.01, 0.001, 0.0001]
    # )
    
    # Use the learning rate schedule in the optimizer
    model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=lr_schedule_ed))
    return model


In [37]:
class RecommendationHyperModel(HyperModel):
    def __init__(self, unique_user_ids, unique_titles, num_genres, rating_weight=1.0, retrieval_weight=1.0):
        self.unique_user_ids = unique_user_ids
        self.unique_titles = unique_titles
        self.num_genres = num_genres
        self.rating_weight = rating_weight
        self.retrieval_weight = retrieval_weight

    def build(self, hp):
        embedding_dimension = hp.Int('embedding_dimension', min_value=32, max_value=256, step=32)
        user_input = tf.keras.layers.Input(shape=(), dtype=tf.int32, name='user_id')
        movie_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='title')
        genre_input = tf.keras.layers.Input(shape=(self.num_genres,), dtype=tf.float32, name='genres')

        user_lookup = tf.keras.layers.IntegerLookup(vocabulary=self.unique_user_ids, mask_token=None)
        movie_lookup = tf.keras.layers.StringLookup(vocabulary=self.unique_titles, mask_token=None)

        user_embedding = tf.keras.layers.Embedding(len(self.unique_user_ids) + 1, embedding_dimension)(user_lookup(user_input))
        movie_embedding = tf.keras.layers.Embedding(len(self.unique_titles) + 1, embedding_dimension)(movie_lookup(movie_input))
        genre_embedding = tf.keras.layers.Dense(embedding_dimension)(genre_input)

        concatenated_embeddings = tf.concat([user_embedding, movie_embedding, genre_embedding], axis=1)

        dense_1 = tf.keras.layers.Dense(hp.Int('units_1', min_value=128, max_value=512, step=64), activation="relu")(concatenated_embeddings)
        dropout_1 = tf.keras.layers.Dropout(hp.Float('dropout_1', min_value=0.1, max_value=0.5, step=0.1))(dense_1)
        dense_2 = tf.keras.layers.Dense(hp.Int('units_2', min_value=64, max_value=256, step=32), activation="relu")(dropout_1)
        dropout_2 = tf.keras.layers.Dropout(hp.Float('dropout_2', min_value=0.1, max_value=0.5, step=0.1))(dense_2)
        rating_output = tf.keras.layers.Dense(1)(dropout_2)

        user_model = tf.keras.Model(inputs=user_input, outputs=user_embedding)
        movie_model = tf.keras.Model(inputs=movie_input, outputs=movie_embedding)
        genre_model = tf.keras.Model(inputs=genre_input, outputs=genre_embedding)
        rating_model = tf.keras.Model(inputs=[user_input, movie_input, genre_input], outputs=rating_output)

        metrics = tfrs.metrics.FactorizedTopK(
            candidates=tf.data.Dataset.from_tensor_slices(self.unique_titles).batch(128).map(movie_model)
        )
        rating_task = tfrs.tasks.Ranking(
            loss=tf.keras.losses.MeanSquaredError(),
            metrics=[tf.keras.metrics.RootMeanSquaredError()],
        )
        retrieval_task = tfrs.tasks.Retrieval(
            metrics=metrics
        )
        model = RecommendationModel(user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, self.rating_weight, self.retrieval_weight)
        
        # Define the learning rate schedule
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log'),
            decay_steps=1000,
            decay_rate=0.9,
            staircase=True
        )
        
        # Use the learning rate schedule in the optimizer
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule))
        return model

In [42]:
tuner = RandomSearch(
    RecommendationHyperModel(unique_userIds, unique_titles, len(unique_genres)),
    objective=Objective("val_factorized_top_k/top_10_categorical_accuracy", direction="max"),
    max_trials=10,
    executions_per_trial=1,
    directory='my_dir',
    project_name='movie_recommendation',
    
)

tuner.search(trainds, epochs=10, validation_data=testds, callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)])

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the best hyperparameters
tuned_model = tuner.hypermodel.build(best_hps)

Trial 10 Complete [00h 20m 23s]
val_factorized_top_k/top_10_categorical_accuracy: 0.19213135540485382

Best val_factorized_top_k/top_10_categorical_accuracy So Far: 0.3572697639465332
Total elapsed time: 03h 56m 57s


In [19]:
# Define callbacks for training
timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
model_export_path = os.path.join(output_dir, 'saved-model', timestamp)
checkpoint_path = os.path.join(output_dir, 'checkpoints', f"{timestamp}_cp.ckpt")
tensorboard_path = os.path.join(output_dir, 'tensorboard', timestamp)
faiss_path = os.path.join(output_dir, 'faiss', f"{timestamp}_faiss.index")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(checkpoint_path),
    save_best_only=True,
    save_weights_only=True,
    monitor='factorized_top_k/top_10_categorical_accuracy',
    mode='max'
)
tensorboard_callback = tf.keras.callbacks.TensorBoard(tensorboard_path, histogram_freq=1)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='factorized_top_k/top_10_categorical_accuracy',
    patience=2,
    restore_best_weights=True
)

In [31]:
model = create_model(unique_userIds, unique_titles, len(unique_genres), rating_weight=1.0, retrieval_weight=1.0)

In [28]:
# Build the model by calling it on a batch of data
model.build(input_shape={
	"user_id": (None,),
	"title": (None,),
	"genres": (None, len(unique_genres))
})

# Print the model summary
print(model.summary())

Model: "recommendation_model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 model_8 (Functional)        (None, 128)               1710848   
                                                                 
 model_9 (Functional)        (None, 128)               678912    
                                                                 
 model_10 (Functional)       (None, 128)               2560      
                                                                 
 model_11 (Functional)       (None, 1)                 2523905   
                                                                 
 ranking_2 (Ranking)         multiple                  0 (unused)
                                                                 
 retrieval_2 (Retrieval)     multiple                  0 (unused)
                                                                 
Total params: 2,523,906
Trainable params: 2,

In [46]:
# Train the model
history = tuned_model.fit(
    trainds,
    epochs=10,
    # steps_per_epoch=5,
    # validation_data=evalds,
    callbacks=[checkpoint_callback, tensorboard_callback, early_stopping_callback]
)

Epoch 1/10


2025-01-20 18:54:26.064965: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 54322 of 200000
2025-01-20 18:54:36.064875: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 110411 of 200000
2025-01-20 18:54:46.065020: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 166338 of 200000
2025-01-20 18:54:47.514298: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.


1091/1091 [==============================] - 176s 131ms/step - root_mean_squared_error: 0.7798 - factorized_top_k/top_1_categorical_accuracy: 0.0197 - factorized_top_k/top_5_categorical_accuracy: 0.1833 - factorized_top_k/top_10_categorical_accuracy: 0.3017 - factorized_top_k/top_50_categorical_accuracy: 0.5579 - factorized_top_k/top_100_categorical_accuracy: 0.6505 - loss: 426.1620 - regularization_loss: 0.0000e+00 - total_loss: 426.1620
Epoch 2/10
1091/1091 [==============================] - 145s 133ms/step - root_mean_squared_error: 0.7521 - factorized_top_k/top_1_categorical_accuracy: 0.0186 - factorized_top_k/top_5_categorical_accuracy: 0.2059 - factorized_top_k/top_10_categorical_accuracy: 0.3345 - factorized_top_k/top_50_categorical_accuracy: 0.6120 - factorized_top_k/top_100_categorical_accuracy: 0.7091 - loss: 387.2874 - regularization_loss: 0.0000e+00 - total_loss: 387.2874
Epoch 3/10
1091/1091 [==============================] - 154s 141ms/step - root_mean_squared_error: 0.72

In [47]:
steps = ds_length * 0.2 // eval_test_batch_size
metrics = tuned_model.evaluate(testds, steps=steps, return_dict=True)

2025-01-20 19:20:52.123979: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 56969 of 200000
2025-01-20 19:21:02.123941: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 113513 of 200000
2025-01-20 19:21:12.123987: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 170673 of 200000
2025-01-20 19:21:12.795082: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:417] Shuffle buffer filled.


545/545 [==============================] - 72s 75ms/step - root_mean_squared_error: 0.6446 - factorized_top_k/top_1_categorical_accuracy: 0.0414 - factorized_top_k/top_5_categorical_accuracy: 0.2448 - factorized_top_k/top_10_categorical_accuracy: 0.3793 - factorized_top_k/top_50_categorical_accuracy: 0.6888 - factorized_top_k/top_100_categorical_accuracy: 0.7934 - loss: 145.3681 - regularization_loss: 0.0000e+00 - total_loss: 145.3681


In [34]:
print(f"Retrieval top-1 accuracy: {metrics['factorized_top_k/top_1_categorical_accuracy']:.3f}.")
print(f"Retrieval top-5 accuracy: {metrics['factorized_top_k/top_5_categorical_accuracy']:.3f}.")
print(f"Retrieval top-10 accuracy: {metrics['factorized_top_k/top_10_categorical_accuracy']:.3f}.")
print(f"Ranking RMSE: {metrics['root_mean_squared_error']:.3f}.")

Retrieval top-1 accuracy: 0.042.
Retrieval top-5 accuracy: 0.224.
Retrieval top-10 accuracy: 0.326.
Ranking RMSE: 0.780.


In [50]:
tuned_model.save_weights(f"trained_model/weights/{timestamp}_weights.h5")

In [51]:
trained_user_embeddings, trained_movie_embeddings, trained_genre_embeddings, predicted_rating = tuned_model({
      "user_id": np.array([56703]),
      "title": np.array(['Siberian Sniper']),
      "genres": np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]])
  })
print("Predicted rating:")
print(predicted_rating)

Predicted rating:
tf.Tensor([[3.4726896]], shape=(1, 1), dtype=float32)


In [52]:
def extract_embeddings(model, unique_user_ids, unique_titles):
    """
    Extract embeddings for all users and movies using the trained model.
    
    Args:
    - model: The trained recommendation model.
    - unique_user_ids: List of unique user IDs.
    - unique_movie_ids: List of unique movie IDs.
    
    Returns:
    - user_embeddings: Embeddings for all users.
    - movie_embeddings: Embeddings for all movies.
    """
    # Extract movie embeddings
    titles = np.array(unique_titles)
    movie_embeddings = model.movie_model(tf.constant(titles, dtype=tf.string)).numpy()

    # Extract user embeddings
    user_ids = np.array(unique_user_ids)
    user_embeddings = model.user_model(tf.constant(user_ids, dtype=tf.int32)).numpy()

    return user_embeddings, movie_embeddings

In [53]:
def index_movie_embeddings(movie_embeddings):
    """
    Index the movie embeddings using FAISS.
    
    Args:
    - movie_embeddings: Embeddings for all movies.
    
    Returns:
    - index: FAISS index with movie embeddings.
    """
    # Dimension of the embeddings
    embedding_dimension = movie_embeddings.shape[1]

    # Create a FAISS index
    index = faiss.IndexFlatL2(embedding_dimension)

    # Add movie embeddings to the index
    index.add(movie_embeddings)

    return index

In [54]:
def recommend_movies(model, index, unique_titles, user_id, k=10):
    """
    Recommend movies for a given user by querying the FAISS index.
    
    Args:
    - model: The trained recommendation model.
    - index: FAISS index with movie embeddings.
    - unique_titles: List of unique movie titles.
    - user_id: The user ID for which to make recommendations.
    - k: Number of recommendations to retrieve (default is 10).
    
    Returns:
    - recommended_movie_ids: List of recommended movie IDs.
    """
    # Get the embedding for the given user
    user_embedding = model.user_model(tf.constant([user_id], dtype=tf.int32)).numpy()

    # Query the FAISS index
    distances, indices = index.search(user_embedding, k)

    # Get the recommended movie IDs
    recommended_movies = np.array(unique_titles)[indices[0]]

    return recommended_movies

In [60]:
# Extract embeddings
user_embeddings, movie_embeddings = extract_embeddings(tuned_model, unique_userIds, unique_titles)

# Index movie embeddings
index = index_movie_embeddings(movie_embeddings)

# Example user ID for which to make recommendations
example_user_id = 56703

recommended_movie_ids = recommend_movies(tuned_model, index, unique_titles, example_user_id, k=3)

print(f"Recommended movie titles for user {example_user_id}: {recommended_movie_ids}")

Recommended movie titles for user 56703: [b'Elephant' b'All Our Fears' b'Animals']


In [61]:
faiss.write_index(index, "trained_model/faiss/{}_faiss.index".format(timestamp))